In [1]:
# required libraries
import xarray as xr
import pandas as pd
import numpy as np

# Extracting Pixel Centre Coordinates

This script extracts the pixel centre coordinates from the file `EVI_time_series_scales_interpolated_nearest_v2.nc`.


In [2]:
# load netcdf fileif not loaded already
ds = xr.open_dataset("/Users/gb4818/Library/CloudStorage/Box-Box/OneDrive To Box Migration/Vegetation-indices/EVI-netcdf/EVI_time_series_scales_interpolated_nearest_v2.nc")
print(ds)

<xarray.Dataset> Size: 60GB
Dimensions:  (time: 574, lat: 3600, lon: 7200)
Coordinates:
  * time     (time) datetime64[ns] 5kB 2000-02-18 2000-03-05 ... 2025-03-26
  * lat      (lat) float64 29kB 89.97 89.92 89.88 89.83 ... -89.87 -89.92 -89.97
  * lon      (lon) float64 58kB -180.0 -179.9 -179.9 ... 179.9 179.9 180.0
Data variables:
    EVI      (time, lat, lon) float32 60GB ...


In [3]:
# extract its components
data = ds['EVI']
lats = ds['lat'].values
lons = ds['lon'].values

chunk_size = 100  # number of latitude rows per chunk
num_chunks = len(lats) // chunk_size + 1

In [ ]:
# Create 2D meshgrid of coordinates
lon_grid, lat_grid = np.meshgrid(lons, lats)
# Flatten and round to 3 decimal places
coords_df = pd.DataFrame({
    'latitude': np.round(lat_grid.ravel(), 3),
    'longitude': np.round(lon_grid.ravel(), 3)
})
# Save so you can use in other scripts
coords_df.to_csv("pixel_centers_global.csv", index=False)


## Filtering the Time Series

Running wavelet analysis on a large number of data points is computationally intensive. To improve performance, we first filter the coordinates in `EVI_time_series_scales_interpolated_nearest_v2.nc` to retain only pixels with complete EVI time series (i.e., no missing values or NaNs).

This script uses the file `EVI_time_series_scales_interpolated_nearest_v2.nc`, which contains only complete EVI time series. The dataset has been previously processed to:

- Enforce a regular 16-day interval between measurements using nearest-neighbour interpolation

- Fill missing values in January 2001 using linear interpolation

The Python script identifies pixels whose EVI time series are complete across all time steps and saves the corresponding pixel centroid coordinates to a CSV file.

In [ ]:
# generate the file
with open("pixels_with_complete_EVI_timeseries.csv", "w") as f:
    f.write("latitude,longitude\n")  # write header once

for i in range(num_chunks):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, len(lats))
    
    # select lat chunk
    data_chunk = data.isel(lat=slice(start, end))
    
    # compute complete mask for chunk
    complete_mask_chunk = data_chunk.notnull().all(dim='time').compute()
    
    # corresponding lat/lon grids
    lat_chunk = lats[start:end]
    lon_grid, lat_grid = np.meshgrid(lons, lat_chunk)
    
    # flatten
    mask_flat = complete_mask_chunk.values.ravel()
    lat_flat = lat_grid.ravel()
    lon_flat = lon_grid.ravel()
    
    # filter pixels with complete time series
    lat_complete = lat_flat[mask_flat]
    lon_complete = lon_flat[mask_flat]
    
    # create df
    df_chunk = pd.DataFrame({
        'latitude': np.round(lat_complete, 3),
        'longitude': np.round(lon_complete, 3),
    })
    
    if np.any(np.isnan(lat_complete)) or np.any(np.isnan(lon_complete)):
        print("Warning: NaNs found in lat/lon output!") #generate warning to double check for NaNs
        print(lat_complete, lon_complete)
    
    # append to CSV
    df_chunk.to_csv("pixels_with_complete_EVI_timeseries.csv", mode='a', index=False, header=False)
    
    # to keep track of progress
    print(f"Processed lat rows {start} to {end}") 


## Slicing
Separate the coordinates in X number subsets so that each subset can be run on HPC as an array job. 

In [ ]:
def split_csv(input_file, output_prefix, num_files):
    # read the input csv file
    df = pd.read_csv(input_file)

    # calculate the chunk size from input number
    chunk_size = len(df) // num_files
    remainder = len(df) % num_files

    # split the dataframe into chunks
    chunks = []
    start = 0
    for i in range(num_files):
        end = start + chunk_size + (1 if i < remainder else 0)
        chunks.append(df[start:end])
        start = end

    # save each chunk as a separate csv file
    for i, chunk in enumerate(chunks):
        output_file = f"./sliced_csvs/points_csv/{output_prefix}_{i + 1}.csv"
        chunk.to_csv(output_file, index=False)

In [ ]:
# run
input_file = "pixels_with_complete_EVI_timeseries.csv"
output_prefix = "points"
num_files = 300

split_csv(input_file, output_prefix, num_files)